# Assignment 7: Web Scraping Speech Corpus, TF-IDF Vectorization & Semantic Similarity Search
**Author:** Kristian Hudák  
**Course:** AI Olympiad (OAI) Preparation Course

This project implements a text-mining and information retrieval system:
1. Web scraping a historical speech archive (`obamaspeeches.com`)
2. Automated HTML parsing, text extraction, and DataFrame curation
3. Feature extraction with `TfidfVectorizer` (scikit-learn)
4. Pairwise semantic similarity search via Cosine Similarity (`cosine_similarity`)
5. Identification and ranking of the most semantically related speeches in the corpus


### Task 1: Scraping Corpus Document URLs
Extracting all internal speech hyperlinks using `requests` and regular expressions.


In [ ]:
import requests
import re

url = "http://obamaspeeches.com/"

html_content = requests.get(url).text


links = re.findall(r'href="([^"]+)"', html_content)


full_links = [link if link.startswith("http") else url.rstrip("/") + "/" + link.lstrip("/") for link in links]



for link in full_links:
    print(link)

print(len(links))

http://obamaspeeches.com/P-Obama-Inaugural-Speech-Inauguration.htm
http://obamaspeeches.com/E11-Barack-Obama-Election-Night-Victory-Speech-Grant-Park-Illinois-November-4-2008.htm
http://obamaspeeches.com/E11-Barack-Obama-Election-Night-Victory-Speech-Grant-Park-Illinois-November-4-2008.htm
http://obamaspeeches.com/E-Barack-Obama-Speech-Manassas-Virgina-Last-Rally-2008-Election.htm
http://obamaspeeches.com/E10-Barack-Obama-The-American-Promise-Acceptance-Speech-at-the-Democratic-Convention-Mile-High-Stadium--Denver-Colorado-August-28-2008.htm
http://obamaspeeches.com/E10-Barack-Obama-The-American-Promise-Acceptance-Speech-at-the-Democratic-Convention-Mile-High-Stadium--Denver-Colorado-August-28-2008.htm
http://obamaspeeches.com/E09-Barack-Obama-Final-Primary-Night-Presumptive-Democratic-Nominee-Speech-St-Paul-Minnesota-June-3-2008.htm
http://obamaspeeches.com/E08-Barack-Obama-North-Carolina-Primary-Night-Raleigh-NC-May-6-2008.htm
http://obamaspeeches.com/E07-Barack-Obama-Pennsylvania-Pr

### Task 2: Speech Corpus Extraction & Preprocessing
Fetching the full HTML for each speech link, extracting sanitized clean text and document titles, and storing the corpus in a Pandas DataFrame.


In [ ]:
import requests
import re
from bs4 import BeautifulSoup
import pandas as pd

speech_data = []

for link in full_links:
    try:
        response = requests.get(link, verify=False)
        response.raise_for_status()
        html_content = response.text


        soup = BeautifulSoup(html_content, "html.parser")


        clean_text = soup.get_text(separator="\n", strip=True)

        clean_text = re.sub("[“ ,.?!:;”’]", " ", clean_text)


        title = soup.title.string if soup.title else link


        speech_data.append({"title": title, "text": clean_text})

    except Exception as e:
        print(f"Chyba pri linku {link}: {e}")


df = pd.DataFrame(speech_data)


text1 = speech_data[0]["text"]


num_words = len(text1.split())
print(num_words)


print(df.head())


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.cafepress.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.amazon.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Chyba pri linku http://www.cafepress.com/obamaquotes: 403 Client Error: Forbidden for url: https://www.cafepress.com/obamaquotes
Chyba pri linku http://www.amazon.com/INSPIRE-NATION-Electrifying-Speeches-Inauguration/dp/0982100531/?tag=obama-speeches-com-20: 500 Server Error: Internal Server Error for url: https://www.amazon.com/INSPIRE-NATION-Electrifying-Speeches-Inauguration/dp/0982100531/?tag=obama-speeches-com-20
3489
                                               title  \
0  Barack Obama Inauguration Speech - Jan 20 Inau...   
1  Barack Obama Speech - Election Night Victory -...   
2  Barack Obama Speech - Election Night Victory -...   
3  Barack Obama Speech - Manassas - Virginia - La...   
4  Barack Obama Speech - Democratic Convention Ac...   

                                                text  
0  Barack Obama Inauguration Speech - Jan 20 Inau...  
1  Barack Obama Speech - Election Night Victory -...  
2  Barack Obama Speech - Election Night Victory -...  
3  Barack Obama 

In [ ]:
pd.DataFrame(speech_data)

,title,text
0,Barack Obama Inauguration Speech - Jan 20 Inau...,Barack Obama Inauguration Speech - Jan 20 Inau...
1,Barack Obama Speech - Election Night Victory -...,Barack Obama Speech - Election Night Victory -...
2,Barack Obama Speech - Election Night Victory -...,Barack Obama Speech - Election Night Victory -...
3,Barack Obama Speech - Manassas - Virginia - La...,Barack Obama Speech - Manassas - Virginia - La...
4,Barack Obama Speech - Democratic Convention Ac...,Barack Obama Speech - Democratic Convention Ac...
...,...,...
107,Obama Speech - Compete Text of Remarks at Tech...,Obama Speech - Compete Text of Remarks at Tech...
108,"Obama Speech - OBAMA ON S.256, THE BANKRUPTCY ...",Obama Speech - OBAMA ON S 256 THE BANKRUPTCY ...
109,Obama Speech - JOHN LEWIS 65TH BIRTHDAY GALA -...,Obama Speech - JOHN LEWIS 65TH BIRTHDAY GALA -...
110,Obama Speech - KEYNOTE ADDRESS AT THE 2004 DEM...,Obama Speech - KEYNOTE ADDRESS AT THE 2004 DEM...


### Task 3: TF-IDF Feature Vectorization
Transforming raw speech text into a TF-IDF term matrix using scikit-learn's `TfidfVectorizer`.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
vectorizer_tfidf = TfidfVectorizer()

In [ ]:
model_tfidf = vectorizer_tfidf.fit_transform(df['text'])

In [ ]:
header_captions = vectorizer_tfidf.get_feature_names_out()
model2_result = pd.DataFrame(model_tfidf.toarray(),
                index=df['title'],
                columns=header_captions)

In [ ]:
model2_result


,000,000003,01,03,05,08,098,10,100,1000,...,yous,youth,youthful,yushchenko,zealous,zero,zeros,zone,zoning,zorin
title,,,,,,,,,,,,,,,,,,,,,
Barack Obama Inauguration Speech - Jan 20 Inaugural,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.002853,0.000000,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Barack Obama Speech - Election Night Victory - Nov 4 2008 - COMPLETE TEXT,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.003357,0.000000,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Barack Obama Speech - Election Night Victory - Nov 4 2008 - COMPLETE TEXT,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.003357,0.000000,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"Barack Obama Speech - Manassas - Virginia - Last Rally of 2008 Election - November 3, 2008",0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.006925,0.000000,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Barack Obama Speech - Democratic Convention Acceptance Speech - COMPLETE TEXT,0.003445,0.0,0.0,0.0,0.0,0.0,0.0,0.001919,0.000000,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Obama Speech - Compete Text of Remarks at TechNet by Barack Obama,0.013927,0.0,0.0,0.0,0.0,0.0,0.0,0.003878,0.000000,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"Obama Speech - OBAMA ON S.256, THE BANKRUPTCY ABUSE AND PREVENTION ACT OF 2005 - Compete Text",0.026898,0.0,0.0,0.0,0.0,0.0,0.0,0.003745,0.020095,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Obama Speech - JOHN LEWIS 65TH BIRTHDAY GALA - Compete Text,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.003373,0.000000,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
cosine_similarity(model2_result)

array([[1.        , 0.88551267, 0.88551267, ..., 0.82743224, 0.83922411,
        0.77633997],
       [0.88551267, 1.        , 1.        , ..., 0.85776094, 0.86424338,
        0.81718249],
       [0.88551267, 1.        , 1.        , ..., 0.85776094, 0.86424338,
        0.81718249],
       ...,
       [0.82743224, 0.85776094, 0.85776094, ..., 1.        , 0.83401767,
        0.79616568],
       [0.83922411, 0.86424338, 0.86424338, ..., 0.83401767, 1.        ,
        0.80201505],
       [0.77633997, 0.81718249, 0.81718249, ..., 0.79616568, 0.80201505,
        1.        ]])

### Task 4: Pairwise Cosine Similarity Analysis & Top Semantic Match Retrieval
Computing the cosine similarity matrix between all speeches, ignoring diagonal self-similarity, and ranking the most closely related pairs.


In [ ]:
cos_sim = cosine_similarity(model2_result)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

similarity = cosine_similarity(model2_result)


tolerance = 1e-10
np.fill_diagonal(similarity, -np.inf)
similarity[similarity > 1.0 - tolerance] = -np.inf


sim_copy = similarity.copy()

for _ in range(3):
    max_val = np.max(sim_copy)

    i, j = np.unravel_index(np.argmax(sim_copy), sim_copy.shape)

    print("Most similar speeches (indices):", i, j, "with cosine similarity:", max_val)


    sim_copy[i, j] = -np.inf
    sim_copy[j, i] = -np.inf



Najpodobnejšie riadky: 57 80 s podobnosťou: 0.9776794257668221
Najpodobnejšie riadky: 65 66 s podobnosťou: 0.9742996278219133
Najpodobnejšie riadky: 39 48 s podobnosťou: 0.973971818407961


In [ ]:
print("Title of speech 1:", df.iloc[57]['title'])
print("Title of speech 2:", df.iloc[80]['title'])
print("Title of speech 3:", df.iloc[65]['title'])
print("Title of speech 4:", df.iloc[66]['title'])
print("Title of speech 5:", df.iloc[39]['title'])
print("Title of speech 6:", df.iloc[48]['title'])



print("Excerpt of speech 3:", df.iloc[65]['text'][:300])


Title prvého prejavu: Obama Speech - 21st Century Schools for a 21st Century Economy - Complete Text
Title druhého prejavu: Obama Speech - Teaching Our Kids in a 21st Century Economy
Title tretieho prejav: Obama Speech - 
Title stvrteho prejav: Obama Speech - Confirmation of Judge Samuel Alito, Jr. Floor Statement- Complete Text
Title piateho prejavu: Obama Speech - Take Back America - Complete Text
Title siesteho prejavu: Obama Speech - EMILY'S List Annual Luncheon - Women in Politics - Complete Text
Title tretieho prejavu: Obama Speech -
Best 
                Speeches of
Barack Obama
through his 2009 Inauguration
Most Recent Speeches are Listed First
• Barack Obama -
Inaugural Speech
• 
                Barack Obama -
Election Night Victory / Presidential Acceptance Speech - Nov 
                4 2008
•
Barack 
                Obama - Night Before the Election - the Last Rally - Manassas 
                Virginia - Nov 3 2008
• 
                Barack Obama - Democratic Nominee Accep